In [ ]:
from datetime import datetime

job_id = session.sql(
    "SELECT UUID_STRING()"
).collect()[0][0]

job_name = "DWH_FACT_MEDICATION_HEALTH_BEHAVIOR_LOAD"
layer_name = "DWH"

start_time = datetime.now()

rows_loaded = 0
rows_failed = 0

run_status = "SUCCESS"
error_message = None

try:

    fact_medications_df = session.sql(f"""

        SELECT

            m.MHP_ID,
            d.MEDICATION_HEALTH_BEHAVIOR_INFO_ID,
            m.HPP_HPP_ID,
            m.MED_MED_ID,
            m.PERSON_PERSON_PATIENT_ID,
            m.PERSON_PERSON_PRESCRIBER_ID,
            m.PRESCRIBER_PCS_PCS_ID,
            m.CONSENT_BY_PERSON_PERSON_ID,
            m.DOSAGE_DESC,
            m.START_DATE,
            m.END_DATE,
            m.DATE_UNKNOWN,
            m.CONSENT_DATE,
            m.CONSENT_COMMENTS,
            m.LAST_REFILL_DATE,
            m.MEDICATION_CMNT,
            m.CLN_RESP_ADVERSE_EFFECT_CMNT,
            m.THRIVE_FAILURE_IND,
            m.ADD_ORGN_ID,
            m.ADD_PERSON_ID,
            m.MOD_ORGN_ID,
            m.MOD_PERSON_ID,
            CURRENT_TIMESTAMP() AS CREATED_DATE,
            m.DELETED_DATE,
            m.CHECKSUM,
            m.ADD_TS,
            m.ADD_USER,
            m.MOD_TS,
            m.MOD_USER

        FROM {STG}.{STG_MEDICATION_HEALTH_BEHAVIOR} m

        INNER JOIN {DWH}.{DIM_MEDICATION_HEALTH_BEHAVIOR_INFO} d
            ON m.PRESCRIBED_FREQUENCY_CODE IS NOT DISTINCT FROM d.PRESCRIBED_FREQUENCY_CODE_AV_ID
            AND m.ADMINISTER_METHOD_CODE IS NOT DISTINCT FROM d.ADMINISTER_METHOD_CODE_AV_ID
            AND m.CONSENT_DECISION_CODE IS NOT DISTINCT FROM d.CONSENT_DECISION_CODE_AV_ID
            AND d.UPDATED_DATE IS NULL

    """)

    fact_medications_df.create_or_replace_temp_view(
        "TEMP_FACT_MEDICATION_HEALTH_BEHAVIOR"
    )

    merge_result = session.sql(f"""

        MERGE INTO {DWH}.{FACT_MEDICATION_HEALTH_BEHAVIOR} tgt
        USING TEMP_FACT_MEDICATION_HEALTH_BEHAVIOR src
            ON tgt.MHP_ID = src.MHP_ID
        WHEN MATCHED AND (
            src.CHECKSUM<>tgt.CHECKSUM
        ) THEN
            UPDATE SET
                MEDICATION_HEALTH_BEHAVIOR_INFO_ID = src.MEDICATION_HEALTH_BEHAVIOR_INFO_ID,
                HPP_HPP_ID = src.HPP_HPP_ID,
                MED_MED_ID = src.MED_MED_ID,
                PERSON_PERSON_PATIENT_ID = src.PERSON_PERSON_PATIENT_ID,
                PERSON_PERSON_PRESCRIBER_ID = src.PERSON_PERSON_PRESCRIBER_ID,
                PRESCRIBER_PCS_PCS_ID = src.PRESCRIBER_PCS_PCS_ID,
                CONSENT_BY_PERSON_PERSON_ID = src.CONSENT_BY_PERSON_PERSON_ID,
                DOSAGE_DESC = src.DOSAGE_DESC,
                START_DATE = src.START_DATE,
                END_DATE = src.END_DATE,
                DATE_UNKNOWN = src.DATE_UNKNOWN,
                CONSENT_DATE = src.CONSENT_DATE,
                CONSENT_COMMENTS = src.CONSENT_COMMENTS,
                LAST_REFILL_DATE = src.LAST_REFILL_DATE,
                MEDICATION_CMNT = src.MEDICATION_CMNT,
                CLN_RESP_ADVERSE_EFFECT_CMNT = src.CLN_RESP_ADVERSE_EFFECT_CMNT,
                THRIVE_FAILURE_IND = src.THRIVE_FAILURE_IND,
                ADD_ORGN_ID = src.ADD_ORGN_ID,
                ADD_PERSON_ID = src.ADD_PERSON_ID,
                MOD_ORGN_ID = src.MOD_ORGN_ID,
                MOD_PERSON_ID = src.MOD_PERSON_ID,
                DELETED_DATE = src.DELETED_DATE,
                CHECKSUM = src.CHECKSUM,
                ADD_TS = src.ADD_TS,
                ADD_USER = src.ADD_USER,
                MOD_TS = src.MOD_TS,
                MOD_USER = src.MOD_USER
        WHEN NOT MATCHED THEN
            INSERT (
                MHP_ID,
                MEDICATION_HEALTH_BEHAVIOR_INFO_ID,
                HPP_HPP_ID,
                MED_MED_ID,
                PERSON_PERSON_PATIENT_ID,
                PERSON_PERSON_PRESCRIBER_ID,
                PRESCRIBER_PCS_PCS_ID,
                CONSENT_BY_PERSON_PERSON_ID,
                DOSAGE_DESC,
                START_DATE,
                END_DATE,
                DATE_UNKNOWN,
                CONSENT_DATE,
                CONSENT_COMMENTS,
                LAST_REFILL_DATE,
                MEDICATION_CMNT,
                CLN_RESP_ADVERSE_EFFECT_CMNT,
                THRIVE_FAILURE_IND,
                ADD_ORGN_ID,
                ADD_PERSON_ID,
                MOD_ORGN_ID,
                MOD_PERSON_ID,
                CREATED_DATE,
                DELETED_DATE,
                CHECKSUM,
                ADD_TS,
                ADD_USER,
                MOD_TS,
                MOD_USER
            )
            VALUES (
                src.MHP_ID,
                src.MEDICATION_HEALTH_BEHAVIOR_INFO_ID,
                src.HPP_HPP_ID,
                src.MED_MED_ID,
                src.PERSON_PERSON_PATIENT_ID,
                src.PERSON_PERSON_PRESCRIBER_ID,
                src.PRESCRIBER_PCS_PCS_ID,
                src.CONSENT_BY_PERSON_PERSON_ID,
                src.DOSAGE_DESC,
                src.START_DATE,
                src.END_DATE,
                src.DATE_UNKNOWN,
                src.CONSENT_DATE,
                src.CONSENT_COMMENTS,
                src.LAST_REFILL_DATE,
                src.MEDICATION_CMNT,
                src.CLN_RESP_ADVERSE_EFFECT_CMNT,
                src.THRIVE_FAILURE_IND,
                src.ADD_ORGN_ID,
                src.ADD_PERSON_ID,
                src.MOD_ORGN_ID,
                src.MOD_PERSON_ID,
                src.CREATED_DATE,
                src.DELETED_DATE,
                src.CHECKSUM,
                src.ADD_TS,
                src.ADD_USER,
                src.MOD_TS,
                src.MOD_USER
            )

    """).collect()

    rows_inserted = merge_result[0][0]
    rows_updated = merge_result[0][1]
    rows_loaded = rows_inserted + rows_updated

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_MEDICATION_HEALTH_BEHAVIOR load completed. Rows Inserted: {rows_inserted}, Rows Updated: {rows_updated}"
    )

    print(
        f"DWH LOAD SUCCESS | Rows Inserted: {rows_inserted} | Rows Updated: {rows_updated}"
    )

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1

    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"FACT_MEDICATION_HEALTH_BEHAVIOR load failed. Error: {error_message}"
    )

    print(
        f"DWH LOAD FAILED: {error_message}"
    )